# 01 — Prepare datasets and generate missingness

This notebook uses the same reusable Python pipeline as `run_experiment.py`. It does not contain a separate implementation of PMAE.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT_CANDIDATES = [Path.cwd(), Path.cwd().parent]
ROOT = next((p.resolve() for p in ROOT_CANDIDATES if (p / "run_experiment.py").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the repository or its notebooks/ directory.")
sys.path.insert(0, str(ROOT / "src"))

from pmae_tabular.data.registry import DATASET_SPECS
from pmae_tabular.experiments import prepare_one, amputate_one
from pmae_tabular.data.preprocessing import load_processed_dataset


## Available datasets

Each key selects one dataset variant. Bike daily/hourly and Wine red/white are separate experiments.

In [ ]:
pd.DataFrame([
    {"dataset": name, "description": spec.description}
    for name, spec in DATASET_SPECS.items()
])

## Choose a dataset and preprocess it

Change `DATASET` to any key shown above. Raw files remain untouched; generated files go to `data/processed/`.

In [ ]:
DATASET = "adult"
processed = prepare_one(DATASET, ROOT, force=False)
{
    "dataset": processed["dataset"],
    "shape": processed["X"].shape,
    "d_numerical": processed["d_numerical"],
    "numerical_columns": processed["numerical_columns"],
    "categorical_columns": processed["categorical_columns"],
    "target_columns": processed["source_metadata"]["excluded_target_columns"],
    "identifier_columns": processed["source_metadata"]["excluded_identifier_columns"],
}

## Generate an amputated dataset

`column_fraction` selects the columns that can contain missing values. `missing_rate` is the missing fraction inside each selected column. The generated mask uses `True` for missing entries.

In [ ]:
MECHANISM = "MCAR"  # MCAR, MAR, or MNAR
COLUMN_FRACTION = 0.30
MISSING_RATE = 0.50
SEED = 42

amputation_file = amputate_one(
    DATASET, ROOT, mechanism=MECHANISM,
    column_fraction=COLUMN_FRACTION, missing_rate=MISSING_RATE, seed=SEED
)
print(amputation_file.relative_to(ROOT))

## Inspect the generated mask

The complete matrix remains in `data/processed/`; only the incomplete matrix and its mask are stored in `data/amputated/`.

In [ ]:
import numpy as np
from pmae_tabular.data.amputation import load_amputation

X_incomplete, missing_mask, amputation_metadata = load_amputation(amputation_file)
print("Incomplete shape:", X_incomplete.shape)
print("Masked cells:", int(missing_mask.sum()))
print("Realized missing fraction:", float(missing_mask.mean()))
print(amputation_metadata)